This script creates Delta tables from raw .txt files that were uploaded in the volume

In [0]:
spark.sql("SELECT current_catalog(), current_schema()").show()
spark.sql("SHOW TABLES").show(truncate=False)

In [0]:
from datetime import datetime
from pyspark.sql.functions import lit, col, to_date

raw_data_volume = "/Volumes/2026_july_balance/bronze_schema/raw_fund_data_files" ## TO UPDATE IN EACH ITERATION
output_schema = "2026_july_balance.bronze_schema"                                ## TO UPDATE IN EACH ITERATION

fund_folders = dbutils.fs.ls(raw_data_volume)

table_dict = {}
# access the folder
for folder in fund_folders:
    files = dbutils.fs.ls(folder.path)
    folder_name = folder.name.replace("/", "").replace("-","_").lower()         ## the folder names contained "-" and "/" which are not allowed in table names

    # Process the files
    for file in files:
        if file.name.endswith('.txt'):
            file_name = file.name.replace('.txt', '').lower()
            table_name = f"{file_name}_{folder_name}"

            try:
                # Detect delimiter
                first_line = spark.read.text(file.path).first().value
                delimiter = "|" if first_line.count('|') > first_line.count('\t') else "\t"
                
                # Read file
                df = spark.read \
                    .option("header", "true") \
                    .option("delimiter", delimiter) \
                    .csv(file.path)

                table_dict[table_name] = {"columns": len(df.columns), "rows": df.count()}
                
                # Add metadata columns for lineage
                df = df.withColumn("fund_id", lit(folder_name)) \
                       .withColumn("source_file", lit(file.name))\
                       .withColumn("process_date", to_date(lit(datetime.now())))
                
                #create table name - folder_name_file_name
                upload_df = f"{output_schema}.{table_name}"
                df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(upload_df)
            
                print(f" Fixed {table_name}: {df.columns}")

            except Exception as e:
                print(f"  FAILED {folder_name}/{file.name}: {e}")
                continue



In [0]:
import pandas as pd
# The first time Jinny ran this and added unnecessary layers. The code above was fixed. Next time just convert the table_dict to a pandas dataframe
table_dict_clean = {k: {"rows": v["rows"], "columns": v["columns"]} for k, v in table_dict.items()}
table_dict_clean

In [0]:
base_path = "/Volumes/workspace/default/uc_fund_data_2025_bronze/"
fund_folders = dbutils.fs.ls(base_path)

# Create bronze schema
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze_fund_data")


for folder in fund_folders:
    files = dbutils.fs.ls(folder.path)
    folder_name = folder.name.replace("/", "").lower()
    

    for file in files:
        if file.name.endswith('.txt'):
            file_name = file.name.replace('.txt', '').lower()
            
            try:
                # Detect delimiter
                first_line = spark.read.text(file.path).first().value
                delimiter = "|" if first_line.count('|') > first_line.count('\t') else "\t"
                
                # Read file
                df = spark.read \
                    .option("header", "true") \
                    .option("delimiter", delimiter) \
                    .csv(file.path)
                
                # Add metadata columns for lineage
                from pyspark.sql.functions import lit
                df = df.withColumn("fund_id", lit(folder_name)) \
                       .withColumn("source_file", lit(file.name)) 
                
                #create table name - folder_name_file_name
                table_name = f"workspace.bronze_fund_data.{folder_name}_{file_name}"
                df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
            
                print(f" Fixed {table_name}: {df.count()} rows, {len(df.columns)} columns")

            
            except Exception as e:
                print(f"  FAILED {folder_name}/{file.name}: {e}")
                continue

print("\n=== Bronze layer fixed ===")

        
